# Chains

## 传统 Chain 使用方式

### 基础链(废弃)

LLMChain 是 LangChain 早期的核心组件，用于将 LLM 和 Prompt 组合成一个可调用的链。

> **注意**: LLMChain 已被标记为遗留API，推荐使用 LCEL（`prompt | llm`）替代。

In [ ]:
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

# 1. 创建大模型实例
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
)

# 2. 创建 PromptTemplate
prompt = PromptTemplate(
    input_variables=["topic"],
    template="请用一句话简单介绍{topic}是什么？",
)

# 3. 创建 LLMChain
chain = LLMChain(llm=llm, prompt=prompt)

# 4. 调用 chain
result = chain.invoke({"topic": "LangChain"})
print("LLMChain 输出:")
print(result["text"])

C:\Users\Administrator\AppData\Local\Temp\ipykernel_33396\634874157.py:23: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


LLMChain 输出:
LangChain 是一个开源的开发框架，它提供了一套工具和接口，帮助开发者轻松地将大型语言模型（LLM）与外部数据源、工具和工作流集成，从而快速构建智能应用。


### 顺序链
- SimpleSequentialChain,表示单个输入输出
- SequentialChian,表示多个输入输出

### 数学链
> LLMMathChain

### 路由链
> RouterChain

### 文档链
> StuffDocumentsChain

## LCEL 现代写法（推荐）

使用 `|` 管道符将 prompt 和 llm 组合成 chain，更简洁直观。

In [2]:
from langchain_core.prompts import ChatPromptTemplate

# LCEL 写法：prompt | llm
prompt_lcel = ChatPromptTemplate.from_template("请用一句话简单介绍{topic}是什么？")
chain_lcel = prompt_lcel | llm

# 调用方式相同
result_lcel = chain_lcel.invoke({"topic": "LangChain"})
print("LCEL 输出:")
print(result_lcel.content)

LCEL 输出:
LangChain是一个帮助开发者快速构建大语言模型（LLM）应用程序的开源框架，它通过提供模块化的组件（如提示管理、记忆和工具集成）来简化开发流程，就像为AI应用搭建了“脚手架”。


## 两种方式对比

| 特性 | LLMChain (传统) | LCEL (推荐) |
|------|----------------|-------------|
| 语法 | `LLMChain(llm=llm, prompt=prompt)` | `prompt \| llm` |
| 可读性 | 较低 | 高 |
| 扩展性 | 需要嵌套 Chain | 管道符组合 |
| 流式支持 | 有限 | 原生支持 |
| 状态 | 已废弃 | 当前推荐 |

## SequentialChain 示例

将多个 Chain 串联执行，前一个的输出作为后一个的输入。

## SimpleSequentialChain 示例

SimpleSequentialChain 是最简单的顺序链，每个步骤只有一个输入和一个输出：
- 输入 → Chain1 → 输出1 → Chain2 → 最终输出

In [3]:
from langchain_classic.chains import SimpleSequentialChain
from langchain_core.prompts import PromptTemplate

# 第一个链：生成标题
prompt_title = PromptTemplate(
    input_variables=["topic"],
    template="请为{topic}生成一个吸引人的标题。",
)
chain_title = LLMChain(llm=llm, prompt=prompt_title)

# 第二个链：根据标题写摘要
prompt_summary = PromptTemplate(
    input_variables=["title"],
    template="请根据标题'{title}'写一段50字以内的摘要。",
)
chain_summary = LLMChain(llm=llm, prompt=prompt_summary)

# 创建 SimpleSequentialChain（单输入单输出）
simple_chain = SimpleSequentialChain(
    chains=[chain_title, chain_summary],
    verbose=True,
)

# 执行：只需要传入最初的输入
result = simple_chain.invoke({"input": "Python编程入门"})
print("最终输出:")
print(result["output"])



> Entering new SimpleSequentialChain chain...
当然！以下是几个供你参考的标题：

1. **《从零开始的Python奇妙之旅：写出你的第一行代码》**
2. **《Python入门宝典：让编程像搭积木一样简单》**
3. **《与Python交个朋友：零基础也能轻松上手的编程指南》**
4. **《代码新世界：Python带你敲开编程的大门》**
5. **《玩转Python：从"Hello World"到你的第一个项目》**
6. **《Python不迷路：小白也能看懂的编程入门课》**
7. **《打开Python的魔法盒：一步一步成为编程达人》**

你可以根据你的具体使用场景（书籍、课程、博客、视频系列等）来选择或调整。需要我帮你进一步润色或定制吗？😊
本文提供多个Python入门教程标题，涵盖零基础友好、循序渐进的实践学习路径，适合编程新手轻松开启代码之旅。

> Finished chain.
最终输出:
本文提供多个Python入门教程标题，涵盖零基础友好、循序渐进的实践学习路径，适合编程新手轻松开启代码之旅。


## SimpleSequentialChain vs SequentialChain

| 特性 | SimpleSequentialChain | SequentialChain |
|------|----------------------|-----------------|
| 输入/输出 | 每步单输入单输出 | 支持多输入多输出 |
| 参数 | `input` | `input_variables` |
| 返回值 | `output` | `output_variables` |
| 适用场景 | 简单串联 | 复杂数据流 |

In [4]:
from langchain_classic.chains import SequentialChain
from langchain_core.prompts import PromptTemplate

# 第一个链：生成大纲
prompt_outline = PromptTemplate(
    input_variables=["topic"],
    template="请为{topic}写一个简短的大纲，包含3个要点。",
)
chain_outline = LLMChain(llm=llm, prompt=prompt_outline, output_key="outline")

# 第二个链：根据大纲写简介
prompt_intro = PromptTemplate(
    input_variables=["outline"],
    template="根据以下大纲写一段简短的介绍：\n{outline}",
)
chain_intro = LLMChain(llm=llm, prompt=prompt_intro, output_key="intro")

# 串联两个链
sequential_chain = SequentialChain(
    chains=[chain_outline, chain_intro],
    input_variables=["topic"],
    output_variables=["outline", "intro"],
    verbose=True,
)

# 执行
result = sequential_chain.invoke({"topic": "Python编程"})
print("大纲:")
print(result["outline"])
print("\n介绍:")
print(result["intro"])



> Entering new SequentialChain chain...

> Finished chain.
大纲:
# Python 编程简要大纲  

## 1. **基础语法与流程控制**  
   - 变量、数据类型、运算符  
   - 条件语句、循环语句  
   - 函数定义与调用  

## 2. **数据结构与文件操作**  
   - 列表、字典、元组等常用结构  
   - 文件读写与异常处理  
   - 模块化与包管理  

## 3. **面向对象与应用扩展**  
   - 类与对象的基本概念  
   - 继承、封装、多态  
   - 常用库介绍（如 `NumPy`、`Pandas`、`requests`）  

---

如果需要进一步展开某个部分，或添加练习项目建议，我可以帮你补充！ 🚀

介绍:
根据您提供的大纲，Python 编程学习通常可分为三个循序渐进的阶段：  

**1. 夯实基础**：从变量、数据类型、运算符等基本语法入手，掌握条件判断、循环等流程控制结构，并学会定义和使用函数，构建扎实的编程思维基础。  

**2. 熟练运用数据与模块**：深入列表、字典、元组等核心数据结构，实践文件读写、异常处理等实用技能，并学会通过模块化和包管理组织代码，提升开发效率。  

**3. 面向对象与生态扩展**：理解类与对象的核心概念，学习继承、封装、多态等面向对象特性，同时熟悉 NumPy（数值计算）、Pandas（数据处理）、requests（网络请求）等热门库，为数据分析、Web 开发等实际应用铺路。  

每个阶段都可配合小型项目练习，例如从计算器、任务列表，到数据分析脚本或简单爬虫，逐步积累实战经验。如需展开某个部分或添加具体练习，欢迎随时告诉我！ 🚀


## LCEL 等效写法

使用 LCEL 的 `RunnablePassthrough` 和 `RunnableLambda` 实现同样的功能。

In [5]:
from langchain_core.runnables import RunnablePassthrough

# LCEL 写法：管道符串联
outline_prompt = ChatPromptTemplate.from_template("请为{topic}写一个简短的大纲，包含3个要点。")
intro_prompt = ChatPromptTemplate.from_template("根据以下大纲写一段简短的介绍：\n{outline}")

# 使用 LCEL 组合
chain_lcel = (
    {"topic": RunnablePassthrough()}
    | outline_prompt
    | llm
    | (lambda x: {"outline": x.content})
    | intro_prompt.partial(topic="Python编程")  # 这里简化处理
)

# 更清晰的 LCEL 写法
def extract_outline(response):
    return {"outline": response.content}

chain_lcel_clear = (
    outline_prompt | llm | extract_outline | intro_prompt | llm
)

result_lcel = chain_lcel_clear.invoke({"topic": "Python编程"})
print("LCEL SequentialChain 输出:")
print(result_lcel.content)

LCEL SequentialChain 输出:
这是一个Python编程入门大纲，旨在帮助你从零开始构建坚实的编程基础。它首先会带你掌握Python的核心语法，比如变量、数据类型和控制流，让你能编写简单的程序。接着，我们将深入常用的数据结构，并探索面向对象编程，学习如何用类和对象来组织更复杂的代码。最后，大纲会介绍NumPy、Pandas等强大库，并演示如何将它们应用于数据处理、可视化及实际开发场景。学完后，你将能独立完成一些实用的小项目，为进阶学习做好准备。


示例：

In [6]:
# pip install -U langchain langchain-community langchain-openai
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")
model = ChatOpenAI(model="gpt-5.5", temperature=0)
chain = create_sql_query_chain(model, db)
response = chain.invoke({"question": "How many employees are there"})

C:\Users\Administrator\AppData\Local\Temp\ipykernel_33396\2254273522.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


BadRequestError: Error code: 400 - {'error': {'code': '400', 'message': 'Param Incorrect', 'param': 'Not supported model gpt-5.5'}}